# 1. Первичный анализ и обработка датафрейма

In [39]:
import pandas as pd
import numpy as np

In [40]:
sber = pd.read_csv(r'C:\Users\user\Desktop\AlgoTrading\data\SBRF.txt')

In [42]:
def good_dataframe(data):
  """Преобразует сырые рыночные данные в чистый DataFrame с правильными типами и индексом времени
    
    Подготавливает данные для технического анализа.
    
    Args:
        data (pd.DataFrame): Исходный DataFrame с рыночными данными, содержащий столбцы:
            ['<TICKER>', '<PER>', '<DATE>', '<TIME>', '<OPEN>', '<HIGH>', '<LOW>', '<CLOSE>', '<VOL>']
            
    Returns:
        tuple: Возвращает кортеж из двух DataFrame:
            - Основной DataFrame
            - Копия DataFrame для безопасного резервирования
            
    Processing Logic:
        1. Удаление избыточных столбцов
        2. Переименование столбцов в human-friendly формат
        3. Преобразование типов данных
        4. Создание правильного временного индекса
    """
  # 1. Удаляем ненужные столбцы (тикер и период не нужны для анализа)
  data.drop(['<TICKER>', '<PER>'], inplace=True, axis=1)
    
  # 2. Переименовываем столбцы для удобства работы
  data.columns = ['date', 'time', 'open', 'high', 'low', 'close', 'volume']
    
  # 3. Преобразуем дату из формата YYYYMMDD в datetime
  data['date'] = pd.to_datetime(data['date'], format='%Y%m%d')
    
  # 4. Обрабатываем время (HHMMSS -> datetime.time)
  data['time'] = pd.to_datetime(data['time'], format='%H%M%S').dt.time
    
  # 5. Комбинируем дату и время в единую метку времени
  data['time'] = pd.to_datetime(
        data['date'].astype('str') + ' ' + data['time'].astype('str'))
    
  # 6. Удаляем отдельный столбец даты (теперь он в индексе)
  data.drop(['date'], inplace=True, axis=1)
  
  # 7. Установка индекса
  data = data.set_index('time')
  
  # 8. Создаем резервную копию для безопасного использования
  data_copy = data.copy()
  
  return data, data_copy


In [43]:
sber_df, sber_df_copy = good_dataframe(sber)

In [44]:
def new_timeframe(data, timeframe):
    """Преобразует минутные данные (1М) в указанный временной интервал, сохраняя структуру OHLCV-данных.
    
    Использует принципы агрегации свечных данных:
    - Open - первое значение периода
    - High - максимум периода
    - Low - минимум периода
    - Close - последнее значение периода
    - Volume - сумма объема за период

    Args:
        data (pd.DataFrame): Исходный DataFrame с 1-минутными данными, 
                            должен содержать колонки ['open', 'high', 'low', 'close', 'volume']
                            и иметь DateTimeIndex
        timeframe (str): Желаемый таймфрейм из списка доступных:
                        ['5 min', '15 min', '30 min', '1h', '2h', '4h', 'D']

    Returns:
        pd.DataFrame: Новый DataFrame с преобразованными данными в указанном таймфрейме
        
    Raises:
        ValueError: Если передан неподдерживаемый timeframe
    """
    dict_tf = {'5 min' : '5min', '15 min' : '15min', '30 min' : '30min',
               '1h' : '1h', '2h' : '2h', '4h' : '4h', 'D' : 'D'}

    return_data = data.resample(dict_tf[timeframe]).agg({
            'open': 'first',
            'high': 'max',
            'low': 'min',
            'close': 'last',
            'volume': 'sum'
        }).dropna()
    
    return return_data


In [45]:
sber_1h = new_timeframe(sber_df, '1h')
sber_1h

,open,high,low,close,volume
time,,,,,
2009-01-11 10:00:00,2301.0,2380.0,2265.0,2361.0,7797
2009-01-11 11:00:00,2366.0,2369.0,2324.0,2346.0,5669
2009-01-11 12:00:00,2346.0,2354.0,2328.0,2339.0,2264
2009-01-11 13:00:00,2335.0,2346.0,2330.0,2338.0,876
2009-01-11 14:00:00,2339.0,2342.0,2333.0,2334.0,860
...,...,...,...,...,...
2025-06-30 19:00:00,29871.0,29892.0,29830.0,29859.0,1093
2025-06-30 20:00:00,29858.0,29879.0,29847.0,29857.0,756
2025-06-30 21:00:00,29857.0,29876.0,29854.0,29865.0,1059


In [ ]:
def changes_of_data_types(data, time_index=False):
    
    """
    Оптимизирует использование памяти в pandas DataFrame путём автоматического преобразования
    типов данных в каждом столбце к наиболее компактному и подходящему формату.
    
    Args:
        data (pd.DataFrame): Исходный DataFrame с данными свечей определённого таймфрейма.
        time_index (bool, optional): Сохраняем ли время, как индекс. Defaults to False.
        - False - убираем временной индекс.
        - True - оставляем временной индекс.

    Returns
        data(pd.DataFRame): DataFrame с оптимизированными типами данных."""
        
    # 1. Убираем временной индекс у таблицы
    if time_index == False:
        data = data.reset_index()
    
    # 2. Проверяем последовательно каждый столбец. Данные будем хранить в float везде, кроме столбца volume.
    # Сначала проверяем столбец volume
    max_volume = np.max(data['volume'])
    if max_volume < 65535:
        data['volume'] = data['volume'].astype('uint16')
    else:
        data['volume'] = data['volume'].astype('uint32')
        
    # Проверяем остальные столбцы
    for colm in ['open', 'high', 'low', 'close']:
        max_r = 0
        for price in colm:
            str_price = str(price).split('.')[1]
            new_str = str_price.rstrip('0')
            
        
    return data
        
        


In [ ]:
numbers = 

In [51]:
result = changes_of_data_types(sber_1h, time_index=True)
result
    

,open,high,low,close,volume
time,,,,,
2009-01-11 10:00:00,2301.0,2380.0,2265.0,2361.0,7797
2009-01-11 11:00:00,2366.0,2369.0,2324.0,2346.0,5669
2009-01-11 12:00:00,2346.0,2354.0,2328.0,2339.0,2264
2009-01-11 13:00:00,2335.0,2346.0,2330.0,2338.0,876
2009-01-11 14:00:00,2339.0,2342.0,2333.0,2334.0,860
...,...,...,...,...,...
2025-06-30 19:00:00,29871.0,29892.0,29830.0,29859.0,1093
2025-06-30 20:00:00,29858.0,29879.0,29847.0,29857.0,756
2025-06-30 21:00:00,29857.0,29876.0,29854.0,29865.0,1059


In [ ]:
def changes_of_data_types(data, time_index=False):
    """_summary_

    Args:
        data (_type_): _description_
        time_index (bool, optional): _description_. Defaults to False.
    """

In [47]:
sber_1h.info()

<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 58525 entries, 2009-01-11 10:00:00 to 2025-06-30 23:00:00
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype  
---  ------  --------------  -----  
 0   open    58525 non-null  float64
 1   high    58525 non-null  float64
 2   low     58525 non-null  float64
 3   close   58525 non-null  float64
 4   volume  58525 non-null  int64  
dtypes: float64(4), int64(1)
memory usage: 2.7 MB
